# Face enrolment + identification + SG-4 multi-face tracking (Class Dataset)

Run cells top to bottom. Runtime > Change runtime type > **T4 GPU**.

- CELL 1–11: enrolment, identification, evaluation, per-frame video test (SG-3 work)
- CELL 12–16: **SG-4 Multi-Person Tracking & Identity Association**

### CELL 1: install (run once)

In [ ]:
!pip install facenet-pytorch --no-deps -q
!pip install pillow-heif -q

### CELL 2: imports and models

In [ ]:
import os, json, random
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont, ImageOps
from facenet_pytorch import MTCNN, InceptionResnetV1
from pillow_heif import register_heif_opener
register_heif_opener()

random.seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

mtcnn = MTCNN(keep_all=True, device=device, min_face_size=20)
embedder = InceptionResnetV1(pretrained="vggface2").eval().to(device)

### CELL 3: settings + mount Drive

In [ ]:
THRESHOLD = 0.60   # cosine similarity; below this -> UNKNOWN (image tests)

from google.colab import drive
drive.mount('/content/drive')

### CELL 4: load Class Dataset

In [ ]:
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/Class Dataset"   # <- your path
print("Path exists:", os.path.exists(DATA_DIR))

by_person = {}
skipped = 0
for name in sorted(os.listdir(DATA_DIR)):
    person_dir = os.path.join(DATA_DIR, name)
    if not os.path.isdir(person_dir) or name == "SG4_out":
        continue
    imgs = []
    for root, _, files in os.walk(person_dir):
        for f in sorted(files):
            try:
                im = ImageOps.exif_transpose(Image.open(os.path.join(root, f))).convert("RGB")
                im.thumbnail((1024, 1024))
                imgs.append(im)
            except Exception as e:
                skipped += 1
                print("SKIPPED:", name, f, "->", type(e).__name__, e)
    by_person[name] = imgs

for n, v in by_person.items():
    print(f"{n:25s} {len(v)} photos")
print("Files skipped:", skipped)

# use all photos for enrolment (accuracy test is done in CELL 10 via leave-one-out)
enroll_set = {n: v for n, v in by_person.items() if len(v) >= 1}
enrolled_names = list(enroll_set.keys())
print("\nEnrolled:", enrolled_names)

### CELL 5: helper functions

In [ ]:
def embed_faces(face_tensors):
    with torch.no_grad():
        e = embedder(face_tensors.to(device))
    return torch.nn.functional.normalize(e, p=2, dim=1).cpu().numpy()

def detect_and_embed(img):
    """-> boxes (N x 4), detection probs (N), embeddings (N x 512) or None"""
    boxes, probs = mtcnn.detect(img)
    if boxes is None:
        return None, None, None
    faces = mtcnn.extract(img, boxes, None)          # N x 3 x 160 x 160
    return boxes, probs, embed_faces(faces)

def largest_face_embedding(img):
    boxes, probs, e = detect_and_embed(img)
    if boxes is None:
        return None
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return e[int(np.argmax(areas))]

for n, ims in by_person.items():
    if not ims:
        print(f"{n:22s} no photos loaded"); continue
    boxes, _ = mtcnn.detect(ims[0])
    print(f"{n:22s} size={ims[0].size}  faces found={0 if boxes is None else len(boxes)}")

### CELL 6: ENROL - give every person an ID, build the gallery

In [ ]:
gallery = {}
for n, name in enumerate(enrolled_names, start=1):
    sid = f"S{n:03d}"
    embs = [largest_face_embedding(im) for im in enroll_set[name]]
    embs = [e for e in embs if e is not None]
    if not embs:
        print(f"[WARN] {name}: no usable face, not enrolled"); continue
    mean = np.mean(embs, axis=0); mean /= np.linalg.norm(mean)
    gallery[sid] = {"name": name, "embedding": mean}
    print(f"{sid} -> {name}  ({len(embs)} images used)")

gal_ids   = list(gallery.keys())
gal_names = [gallery[i]["name"] for i in gal_ids]
gal_mat   = np.stack([gallery[i]["embedding"] for i in gal_ids])

np.savez("gallery.npz", ids=gal_ids, names=gal_names, embs=gal_mat)
json.dump({i: gallery[i]["name"] for i in gal_ids}, open("id_table.json", "w"), indent=2)
print(f"\nEnrolled {len(gallery)} people. Saved gallery.npz, id_table.json")

### CELL 7: identification + drawing functions

In [ ]:
def identify(img, threshold=None):
    threshold = THRESHOLD if threshold is None else threshold
    boxes, probs, embs = detect_and_embed(img)
    results = []
    if boxes is None:
        return results
    sims = embs @ gal_mat.T
    for b, p, s in zip(boxes, probs, sims):
        j = int(np.argmax(s))
        known = s[j] >= threshold
        results.append({"box": b.tolist(), "det_conf": float(p),
                        "id": gal_ids[j] if known else "UNKNOWN",
                        "name": gal_names[j] if known else "UNKNOWN",
                        "score": float(s[j])})
    return results

def draw_results(img, results):
    out = img.copy(); d = ImageDraw.Draw(out)
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 14)
    except Exception:
        font = ImageFont.load_default()
    for r in results:
        x1, y1, x2, y2 = r["box"]
        color = (0, 200, 0) if r["id"] != "UNKNOWN" else (220, 0, 0)
        d.rectangle([x1, y1, x2, y2], outline=color, width=3)
        text = "UNKNOWN" if r["name"] == "UNKNOWN" else f'{r["name"]} {r["score"]:.2f}'
        d.text((x1, max(0, y1 - 16)), text, fill=color, font=font)
    return out

def show(img, title=""):
    plt.figure(figsize=(10, 7)); plt.imshow(img); plt.axis("off")
    plt.title(title); plt.show()

### CELL 8: TEST 1 - single image of an enrolled person

In [ ]:
name = enrolled_names[0]
img = enroll_set[name][0]
res = identify(img)
print("Ground truth:", name)
for r in res:
    print(r)
show(draw_results(img, res), f"Single image test (truth: {name})")

### CELL 9: TEST 2 - multi-face collage

In [ ]:
pick = enrolled_names[:6]
items = [(enroll_set[n][-1], n) for n in pick]
random.shuffle(items)

def make_collage(items, cols=3, size=250):
    rows = (len(items) + cols - 1) // cols
    canvas = Image.new("RGB", (cols * size, rows * size), (255, 255, 255))
    for k, (im, _) in enumerate(items):
        canvas.paste(im.resize((size, size)), ((k % cols) * size, (k // cols) * size))
    return canvas

collage = make_collage(items)
res = identify(collage)
print(f"Faces in collage: {len(items)} | faces detected: {len(res)}\n")
print("Ground truth (row-major):", [n for _, n in items], "\n")
for i, r in enumerate(res, 1):
    print(f'{i}. {r["id"]:8s} {r["name"]:22s} score={r["score"]:.3f}')
show(draw_results(collage, res), "Multi-face test")

### CELL 10: EVALUATION - leave-one-out accuracy + threshold sweep

In [ ]:
embs_all = {}
for n, ims in by_person.items():
    e = [largest_face_embedding(im) for im in ims]
    embs_all[n] = [x for x in e if x is not None]
    print(f"{n:22s} {len(embs_all[n])}/{len(ims)} faces detected")

def template(vecs):
    m = np.mean(vecs, axis=0)
    return m / np.linalg.norm(m)

names = [n for n in embs_all if len(embs_all[n]) >= 2]
full = {n: template(embs_all[n]) for n in names}
known, unknown = [], []

for n in names:                      # KNOWN: leave one photo out of its owner's template
    for k in range(len(embs_all[n])):
        rest = embs_all[n][:k] + embs_all[n][k+1:]
        gal = dict(full); gal[n] = template(rest)
        s = {m: float(gal[m] @ embs_all[n][k]) for m in gal}
        b = max(s, key=s.get); known.append((n, b, s[b]))

for n in names:                      # UNKNOWN: remove the person from the gallery
    gal = {m: full[m] for m in names if m != n}
    for e in embs_all[n]:
        s = {m: float(gal[m] @ e) for m in gal}
        b = max(s, key=s.get); unknown.append((n, b, s[b]))

print(f"\n{'thr':>5} | {'correct':>8} | {'false rej':>9} | {'wrong ID':>8} | {'unk rejected':>12} | {'false accept':>12}")
for thr in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    c  = sum(1 for t, b, s in known if s >= thr and b == t)
    fr = sum(1 for t, b, s in known if s < thr)
    w  = len(known) - c - fr
    ur = sum(1 for t, b, s in unknown if s < thr)
    print(f"{thr:5.1f} | {c/len(known):8.1%} | {fr/len(known):9.1%} | {w/len(known):8.1%} | "
          f"{ur/len(unknown):12.1%} | {(len(unknown)-ur)/len(unknown):12.1%}")

### CELL 11: per-frame recognition on a 20 s video snippet (SG-3 only, no tracking)

In [ ]:
import cv2
mtcnn = MTCNN(keep_all=True, device=device, min_face_size=15)   # smaller faces in wide shot

video_path  = "/content/drive/MyDrive/Colab Notebooks/Class Dataset/CR # 29 - Tuesday 6 Oct 2026 - 1155 to 1305.mp4"
output_path = "/content/drive/MyDrive/Colab Notebooks/Class Dataset/PROCESSED_20SEC_NAMES.mp4"
NEW_THRESHOLD = 0.45

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print(f"Error: Could not open video at {video_path}")
else:
    fps = cap.get(cv2.CAP_PROP_FPS)
    width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    max_frames = int(fps * 20)
    cap.set(cv2.CAP_PROP_POS_FRAMES, total_frames // 2)
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret or frame_count >= max_frames:
            break
        pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        results = identify(pil_img, threshold=NEW_THRESHOLD)
        out.write(cv2.cvtColor(np.array(draw_results(pil_img, results)), cv2.COLOR_RGB2BGR))
        frame_count += 1
        if frame_count % (int(fps) * 5) == 0:
            print(f"Processed {frame_count} / {max_frames} frames")
    cap.release(); out.release()
    print(f"Done! Saved to {output_path}")

# SG-4: Multi-Person Tracking & Identity Association

**Interface:** detections (SG-1) + identity evidence (SG-3) → persistent Track ID + identity.

- Kalman filter + ByteTrack two-stage IoU association (Hungarian) → persistent Track IDs
- Recognition only on confirmed tracks, gated by quality (size / det-conf / blur), every 3 frames until locked, then every 15 frames to re-verify
- Track-level identity = mean embedding vs gallery; lock after ≥3 obs, score ≥ thr, top1–top2 margin; unlock on 4 consecutive contradicting obs (ID swap)
- Re-entry: lost tracks kept for 300 frames; a new track with a matching face inherits the old Track ID
- Conflict: same student locked on two tracks → higher score keeps it, other is reset
- Outputs: `sg4_frame_log.csv`, `sg4_tracks_mot.txt`, `sg4_tracks.csv` (→ SG-5), `sg4_stats.json`, annotated video

### CELL 12: SG-4 config + tracker

In [ ]:
import time, csv
import pandas as pd
from collections import defaultdict
from scipy.optimize import linear_sum_assignment

mtcnn = MTCNN(keep_all=True, device=device, min_face_size=15)

P = dict(
    DET_HIGH=0.95, DET_LOW=0.70,     # ByteTrack two-stage (MTCNN probs are >=0.70)
    IOU_HIGH=0.30, IOU_LOW=0.20,     # IoU gates for stage 1 / stage 2
    MIN_HITS=3,  MAX_AGE=30,         # confirm after 3 hits; keep lost track 30 frames
    RECOG_UNLOCKED=3, RECOG_LOCKED=15,   # recognition interval (frames)
    MIN_FACE=20, MIN_BLUR=20.0,      # quality gate before recognition
    ID_THR=0.45, ID_MARGIN=0.05,     # track-level accept threshold + top1-top2 margin
    LOCK_MIN_OBS=3, UNLOCK_OBS=4,    # lock after 3 obs; unlock after 4 contradicting obs
    REID_THR=0.55, REID_WINDOW=300,  # re-entry merge threshold / memory (frames)
)

# ---------- geometry ----------
def iou_matrix(a, b):
    a = np.asarray(a, float).reshape(-1, 4); b = np.asarray(b, float).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda r: (r[:, 2] - r[:, 0]) * (r[:, 3] - r[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def associate(tboxes, dboxes, thr):
    iou = iou_matrix(tboxes, dboxes)
    if iou.size == 0: return [], list(range(len(tboxes))), list(range(len(dboxes)))
    r, c = linear_sum_assignment(-iou)
    m = [(i, j) for i, j in zip(r, c) if iou[i, j] >= thr]
    mt, md = {i for i, _ in m}, {j for _, j in m}
    return m, [i for i in range(len(tboxes)) if i not in mt], [j for j in range(len(dboxes)) if j not in md]

# ---------- Kalman filter: state [cx, cy, w, h, vcx, vcy, vw, vh] ----------
class KF:
    def __init__(self, box):
        x1, y1, x2, y2 = box
        self.x = np.array([(x1+x2)/2, (y1+y2)/2, x2-x1, y2-y1, 0, 0, 0, 0], float)
        self.P = np.diag([10, 10, 10, 10, 100, 100, 100, 100]).astype(float)
        self.F = np.eye(8); self.F[:4, 4:] = np.eye(4)
        self.H = np.eye(4, 8)
        self.Q = np.diag([1, 1, 1, 1, .5, .5, .5, .5])
        self.R = np.diag([4, 4, 8, 8]).astype(float)
    def predict(self):
        self.x = self.F @ self.x; self.P = self.F @ self.P @ self.F.T + self.Q
        self.x[2:4] = np.maximum(self.x[2:4], 1); return self.box()
    def update(self, box):
        x1, y1, x2, y2 = box
        z = np.array([(x1+x2)/2, (y1+y2)/2, x2-x1, y2-y1])
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ (z - self.H @ self.x); self.P = (np.eye(8) - K @ self.H) @ self.P
    def box(self):
        cx, cy, w, h = self.x[:4]; return np.array([cx-w/2, cy-h/2, cx+w/2, cy+h/2])

# ---------- quality gate (SG-2 style) ----------
def face_quality_ok(gray, box, conf, P):
    H, W = gray.shape
    x1, y1, x2, y2 = [int(round(v)) for v in box]
    x1, y1, x2, y2 = max(0, x1), max(0, y1), min(W, x2), min(H, y2)
    if x2-x1 < P["MIN_FACE"] or y2-y1 < P["MIN_FACE"] or conf < P["DET_HIGH"]: return False
    return cv2.Laplacian(gray[y1:y2, x1:x2], cv2.CV_64F).var() >= P["MIN_BLUR"]

# ---------- track with identity evidence ----------
class Track:
    _next_id = 1
    def __init__(self, box, conf, f):
        self.tid = Track._next_id; Track._next_id += 1
        self.kf = KF(box); self.box = np.asarray(box, float); self.conf = float(conf)
        self.hits, self.miss, self.confirmed = 1, 0, False
        self.first = self.last = f; self.n_frames = 1
        self.emb_sum, self.n_obs, self.n_recog, self.last_recog = None, 0, 0, -10**9
        self.sid, self.score, self.locked, self.contra = "UNKNOWN", 0.0, False, 0
        self.banned, self.reid_checked = set(), False

    def mean_emb(self):
        return self.emb_sum / (np.linalg.norm(self.emb_sum) + 1e-9)

    def needs_recog(self, f, P):
        gap = P["RECOG_LOCKED"] if (self.locked or self.n_obs >= 20) else P["RECOG_UNLOCKED"]
        return f - self.last_recog >= gap

    def add_obs(self, e, f, P):
        self.n_recog += 1; self.last_recog = f
        self.emb_sum = e.copy() if self.emb_sum is None else self.emb_sum + e
        self.n_obs += 1
        s_now = gal_mat @ e
        mean_sim = gal_mat @ self.mean_emb()
        if self.banned: mean_sim[list(self.banned)] = -1
        order = np.argsort(mean_sim)[::-1]
        top = mean_sim[order[0]]; second = mean_sim[order[1]] if len(order) > 1 else -1
        if not self.locked:
            self.score = float(top)
            if self.n_obs >= P["LOCK_MIN_OBS"] and top >= P["ID_THR"] and top - second >= P["ID_MARGIN"]:
                self.sid, self.locked = gal_ids[order[0]], True
        else:
            k = gal_ids.index(self.sid); self.score = float(mean_sim[k])
            j = int(np.argmax(s_now))
            contra = j != k and s_now[j] >= P["ID_THR"] and s_now[j] - s_now[k] >= P["ID_MARGIN"]
            self.contra = self.contra + 1 if contra else 0
            if self.contra >= P["UNLOCK_OBS"]:          # tracker swapped faces -> restart evidence
                self.sid, self.locked, self.contra = "UNKNOWN", False, 0
                self.emb_sum, self.n_obs, self.score = e.copy(), 1, float(s_now.max())

# ---------- multi-face tracker ----------
class FaceTracker:
    def __init__(self, P):
        self.P, self.tracks, self.dead, self.f = P, [], [], 0
        self.registry, self.stats, self.merges = {}, defaultdict(int), []
        Track._next_id = 1

    def step(self, img_pil, gray):
        P = self.P; self.f += 1; f = self.f
        boxes, probs = mtcnn.detect(img_pil)
        if boxes is None: boxes, probs = np.zeros((0, 4)), np.zeros(0)
        probs = np.asarray(probs, float); keep = probs >= P["DET_LOW"]
        boxes, probs = boxes[keep], probs[keep]
        self.stats["detections"] += len(boxes)

        for t in self.tracks: t.box = t.kf.predict()
        hi = np.where(probs >= P["DET_HIGH"])[0]; lo = np.where(probs < P["DET_HIGH"])[0]

        # stage 1: all tracks <-> high-conf detections
        m1, ut, ud_hi = associate([t.box for t in self.tracks], boxes[hi], P["IOU_HIGH"])
        matched = [(self.tracks[i], hi[j]) for i, j in m1]
        # stage 2: leftover confirmed tracks <-> low-conf detections
        rem = [self.tracks[i] for i in ut if self.tracks[i].confirmed]
        m2, _, _ = associate([t.box for t in rem], boxes[lo], P["IOU_LOW"])
        matched += [(rem[i], lo[j]) for i, j in m2]
        mset = {id(t) for t, _ in matched}

        for t, d in matched:
            t.kf.update(boxes[d]); t.box = boxes[d].astype(float); t.conf = float(probs[d])
            t.hits += 1; t.miss = 0; t.last = f; t.n_frames += 1
            if not t.confirmed and t.hits >= P["MIN_HITS"]:
                t.confirmed = True; self.registry[t.tid] = t

        alive = []
        for t in self.tracks:
            if id(t) in mset: alive.append(t); continue
            t.miss += 1
            if not t.confirmed: continue                              # tentative miss -> drop
            if t.miss > P["MAX_AGE"]:
                if t.emb_sum is not None: self.dead.append(t)         # keep for re-entry
                continue
            alive.append(t)
        alive += [Track(boxes[hi[j]], probs[hi[j]], f) for j in ud_hi]
        self.tracks = alive
        self.dead = [t for t in self.dead if f - t.last <= P["REID_WINDOW"]]

        # selective recognition (only confirmed, visible, due, good quality)
        cand = [t for t in self.tracks if t.confirmed and t.miss == 0
                and t.needs_recog(f, P) and face_quality_ok(gray, t.box, t.conf, P)]
        if cand:
            embs = embed_faces(mtcnn.extract(img_pil, np.stack([t.box for t in cand]), None))
            self.stats["recog_calls"] += len(cand)
            for t, e in zip(cand, embs):
                if not t.reid_checked:
                    t.reid_checked = True; self._try_reid(t, e)
                t.add_obs(e, f, P)
        self._resolve_conflicts()
        return [t for t in self.tracks if t.confirmed and t.miss == 0]

    def _try_reid(self, t, e):
        if not self.dead: return
        active = {x.tid for x in self.tracks}
        sims = [float(d.mean_emb() @ e) for d in self.dead]
        k = int(np.argmax(sims))
        if sims[k] < self.P["REID_THR"] or self.dead[k].tid in active: return
        d = self.dead.pop(k)
        self.registry.pop(t.tid, None); self.merges.append((d.tid, t.tid, self.f, sims[k]))
        t.tid, t.first = d.tid, d.first
        t.n_frames += d.n_frames; t.n_recog += d.n_recog
        t.emb_sum, t.n_obs = d.emb_sum.copy(), d.n_obs
        t.sid, t.locked, t.score, t.banned = d.sid, d.locked, d.score, set(d.banned)
        self.registry[t.tid] = t

    def _resolve_conflicts(self):
        by_sid = defaultdict(list)
        for t in self.tracks:
            if t.locked: by_sid[t.sid].append(t)
        for sid, ts in by_sid.items():
            if len(ts) < 2: continue
            ts.sort(key=lambda t: (t.score, t.n_obs), reverse=True)
            for t in ts[1:]:
                t.banned.add(gal_ids.index(sid))
                t.sid, t.locked, t.contra = "UNKNOWN", False, 0
                self.stats["conflicts_resolved"] += 1

def draw_tracks(frame, tracks, info=""):
    for t in tracks:
        x1, y1, x2, y2 = map(int, t.box)
        col = (0, 200, 0) if t.locked else ((0, 200, 255) if t.n_obs else (0, 0, 220))
        name = gallery[t.sid]["name"] if t.locked else "UNKNOWN"
        cv2.rectangle(frame, (x1, y1), (x2, y2), col, 2)
        cv2.putText(frame, f"T{t.tid} {name[:18]} {t.score:.2f}", (x1, max(12, y1-5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, col, 1, cv2.LINE_AA)
    if info: cv2.putText(frame, info, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    return frame

### CELL 13: run SG-4 on the class video + export outputs
Green = identity locked · Yellow = evidence collecting · Red = no recognition yet

In [ ]:
VIDEO   = "/content/drive/MyDrive/Colab Notebooks/Class Dataset/CR # 29 - Tuesday 6 Oct 2026 - 1155 to 1305.mp4"
OUT_DIR = "/content/drive/MyDrive/Colab Notebooks/Class Dataset/SG4_out"
START_SEC, DURATION_SEC = None, 20      # None = middle of video; DURATION_SEC=None -> to end
os.makedirs(OUT_DIR, exist_ok=True)

def run_sg4(video, start_sec, duration_sec, P, out_dir, save_video=True, save_clip=True):
    cap = cv2.VideoCapture(video); assert cap.isOpened(), video
    fps = cap.get(cv2.CAP_PROP_FPS)
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    start = int(start_sec * fps) if start_sec is not None else total // 2
    n_max = int(duration_sec * fps) if duration_sec else total - start
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    vw = cv2.VideoWriter(os.path.join(out_dir, "sg4_tracked.mp4"), fourcc, fps, (W, H)) if save_video else None
    cw = cv2.VideoWriter(os.path.join(out_dir, "sg4_clip_raw.mp4"), fourcc, fps, (W, H)) if save_clip else None

    trk, rows, t0 = FaceTracker(P), [], time.time()
    while trk.f < n_max:
        ok, frame = cap.read()
        if not ok: break
        if cw: cw.write(frame)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        vis = trk.step(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)), gray)
        for t in vis:
            x1, y1, x2, y2 = t.box
            rows.append([trk.f, t.tid, x1, y1, x2-x1, y2-y1, t.conf, t.sid, t.score])
        if vw:
            vw.write(draw_tracks(frame, vis, f"f{trk.f} tracks={len(vis)} {trk.f/(time.time()-t0):.1f} FPS"))
        if trk.f % int(fps * 5) == 0: print(f"{trk.f}/{n_max} frames")
    elapsed = time.time() - t0
    cap.release()
    if vw: vw.release()
    if cw: cw.release()

    to_s = lambda fr: (start + fr - 1) / fps
    final = {tid: t.sid for tid, t in trk.registry.items()}
    for r in rows: r.append(final.get(r[1], r[7]))

    # per-frame log
    with open(os.path.join(out_dir, "sg4_frame_log.csv"), "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["frame", "time_s", "track_id", "x", "y", "w", "h", "det_conf", "sid_live", "score_live", "sid_final"])
        for r in rows:
            w.writerow([r[0], round(to_s(r[0]), 2), r[1], *np.round(r[2:6], 1), round(r[6], 3), r[7], round(r[8], 3), r[9]])
    # MOT format (for motmetrics)
    with open(os.path.join(out_dir, "sg4_tracks_mot.txt"), "w") as fh:
        for r in rows: fh.write(f"{r[0]},{r[1]},{r[2]:.1f},{r[3]:.1f},{r[4]:.1f},{r[5]:.1f},{r[6]:.3f},-1,-1,-1\n")
    # track summary -> SG-5 interface
    df = pd.DataFrame([dict(
        track_id=tid, student_id=t.sid, name=gallery[t.sid]["name"] if t.locked else "UNKNOWN",
        confidence=round(t.score, 3), locked=t.locked,
        first_seen_s=round(to_s(t.first), 2), last_seen_s=round(to_s(t.last), 2),
        visible_frames=t.n_frames, visible_s=round(t.n_frames / fps, 2),
        n_obs=t.n_obs, n_recog=t.n_recog) for tid, t in sorted(trk.registry.items())])
    df.to_csv(os.path.join(out_dir, "sg4_tracks.csv"), index=False)

    # stats
    by_tid, by_f = defaultdict(list), defaultdict(list)
    for r in rows:
        if r[7] != "UNKNOWN": by_tid[r[1]].append(r[7]); by_f[r[0]].append(r[7])
    det, calls = trk.stats["detections"], trk.stats["recog_calls"]
    stats = dict(
        frames=trk.f, proc_fps=round(trk.f / elapsed, 2), detections=det, recog_calls=calls,
        recog_reduction=round(1 - calls / max(det, 1), 3),
        confirmed_tracks=len(trk.registry), locked_tracks=int(sum(t.locked for t in trk.registry.values())),
        students_identified=len({t.sid for t in trk.registry.values() if t.locked}),
        identity_switches=int(sum(sum(a != b for a, b in zip(s, s[1:])) for s in by_tid.values())),
        duplicate_id_frames=int(sum(len(v) != len(set(v)) for v in by_f.values())),
        reentry_merges=len(trk.merges), conflicts_resolved=trk.stats["conflicts_resolved"])
    json.dump(stats, open(os.path.join(out_dir, "sg4_stats.json"), "w"), indent=2)
    return stats, df

sg4_stats, sg4_tracks = run_sg4(VIDEO, START_SEC, DURATION_SEC, P, OUT_DIR)
print(json.dumps(sg4_stats, indent=2)); display(sg4_tracks)

### CELL 14: baseline (recognise every face every frame) vs SG-4

In [ ]:
def run_baseline(video, start_sec, duration_sec, thr):
    cap = cv2.VideoCapture(video); fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    start = int(start_sec * fps) if start_sec is not None else total // 2
    n_max = int(duration_sec * fps) if duration_sec else total - start
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    n = calls = dup = unk = 0; t0 = time.time()
    while n < n_max:
        ok, frame = cap.read()
        if not ok: break
        n += 1
        res = identify(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)), threshold=thr)
        calls += len(res); ids = [r["id"] for r in res if r["id"] != "UNKNOWN"]
        dup += len(ids) != len(set(ids)); unk += len(res) - len(ids)
    cap.release()
    return dict(frames=n, proc_fps=round(n / (time.time() - t0), 2), recog_calls=calls,
                duplicate_id_frames=dup, unknown_rate=round(unk / max(calls, 1), 3))

base = run_baseline(VIDEO, START_SEC, DURATION_SEC, P["ID_THR"])
print(pd.DataFrame({"Per-frame (SG-3 only)": base,
                    "SG-4 tracked": {k: sg4_stats.get(k, "-") for k in base}}))

### CELL 15: MOT metrics vs ground truth
1. Upload `SG4_out/sg4_clip_raw.mp4` to CVAT, draw face boxes with one track ID per student.
2. Export as **MOT 1.1**, put `gt.txt` in `SG4_out/gt/` (frame numbers match the clip).

In [ ]:
!pip install motmetrics -q
import motmetrics as mm
if not hasattr(np, "asfarray"): np.asfarray = lambda a, dtype=float: np.asarray(a, dtype=dtype)

gt = mm.io.loadtxt(os.path.join(OUT_DIR, "gt/gt.txt"), fmt="mot15-2D", min_confidence=1)
ts = mm.io.loadtxt(os.path.join(OUT_DIR, "sg4_tracks_mot.txt"), fmt="mot15-2D")
acc = mm.utils.compare_to_groundtruth(gt, ts, "iou", distth=0.5)
mh = mm.metrics.create()
summ = mh.compute(acc, name="SG4", metrics=[
    "num_frames", "mota", "motp", "idf1", "idp", "idr", "num_switches",
    "num_fragmentations", "mostly_tracked", "mostly_lost", "precision", "recall"])
print(mm.io.render_summary(summ, formatters=mh.formatters, namemap=mm.io.motchallenge_metric_names))

### CELL 16 (optional): parameter sweep for the report

In [ ]:
for rl in [5, 15, 30]:
    for thr in [0.40, 0.45, 0.50]:
        Pk = dict(P, RECOG_LOCKED=rl, ID_THR=thr)
        s, _ = run_sg4(VIDEO, START_SEC, DURATION_SEC, Pk, OUT_DIR, save_video=False, save_clip=False)
        print(f"RECOG_LOCKED={rl:2d} ID_THR={thr:.2f} | calls={s['recog_calls']:5d} "
              f"red={s['recog_reduction']:.1%} fps={s['proc_fps']:5.1f} "
              f"locked={s['locked_tracks']:3d} students={s['students_identified']:2d} "
              f"idsw={s['identity_switches']} reentry={s['reentry_merges']}")